In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
import json

DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

SEED = 42

In [3]:
# Carrega os metadados processados
df = pd.read_csv(PROCESSED_DIR / "metadata_labeled.csv")
print(f"Total de imagens: {len(df)}")
df.head()

Total de imagens: 10015


,lesion_id,image_id,dx,dx_type,age,sex,localization,label_name
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,Benign Keratosis
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,Benign Keratosis
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,Benign Keratosis
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,Benign Keratosis
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,Benign Keratosis


In [4]:
# Imputação de valores nulos na coluna 'age' com a mediana
age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median)

print(f"Mediana de idade usada para imputação: {age_median}")
print(f"Nulos restantes:\n{df.isnull().sum()}")

Mediana de idade usada para imputação: 50.0
Nulos restantes:
lesion_id       0
image_id        0
dx              0
dx_type         0
age             0
sex             0
localization    0
label_name      0
dtype: int64


In [5]:
# Mapeamento de classes para números inteiros
label_map = {
    'nv':    0,
    'mel':   1,
    'bkl':   2,
    'bcc':   3,
    'akiec': 4,
    'vasc':  5,
    'df':    6
}

df['label'] = df['dx'].map(label_map)

print("Mapeamento de classes:")
for k, v in label_map.items():
    print(f"  {v} → {k}")

Mapeamento de classes:
  0 → nv
  1 → mel
  2 → bkl
  3 → bcc
  4 → akiec
  5 → vasc
  6 → df


In [6]:
# Cria a coluna 'filepath' com o caminho completo das imagens
img_dir = DATA_DIR / "images"
df['filepath'] = df['image_id'].apply(lambda x: str(img_dir / f"{x}.jpg"))

# Verificar se todas as imagens existem
missing = df[~df['filepath'].apply(lambda x: Path(x).exists())]
print(f"Imagens faltando: {len(missing)}")

Imagens faltando: 0


In [7]:
# Split dos dados em treino, validação e teste, garantindo que imagens do mesmo paciente 
# (lesion_id) não apareçam em mais de um conjunto

# Primeiro split: separa teste (15%) do restante (85%)
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
remaining_idx, test_idx = next(gss_test.split(df, df['label'], groups=df['lesion_id']))

df_remaining = df.iloc[remaining_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

# Segundo split: separa validação (15% do restante) de treino
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_idx, val_idx = next(gss_val.split(df_remaining, df_remaining['label'], groups=df_remaining['lesion_id']))

df_train = df_remaining.iloc[train_idx].reset_index(drop=True)
df_val = df_remaining.iloc[val_idx].reset_index(drop=True)

print(f"Treino:    {len(df_train)} imagens ({len(df_train)/len(df)*100:.1f}%)")
print(f"Validação: {len(df_val)} imagens ({len(df_val)/len(df)*100:.1f}%)")
print(f"Teste:     {len(df_test)} imagens ({len(df_test)/len(df)*100:.1f}%)")

Treino:    7182 imagens (71.7%)
Validação: 1306 imagens (13.0%)
Teste:     1527 imagens (15.2%)


In [8]:
# Verifica vazamento de dados entre os conjuntos
# Valores diferentes de 0 indicam que há imagens do mesmo paciente em mais de um conjunto
train_lesions = set(df_train['lesion_id'])
val_lesions = set(df_val['lesion_id'])
test_lesions = set(df_test['lesion_id'])

print(f"Vazamento treino/val:   {len(train_lesions & val_lesions)} lesion_ids em comum")
print(f"Vazamento treino/teste: {len(train_lesions & test_lesions)} lesion_ids em comum")
print(f"Vazamento val/teste:    {len(val_lesions & test_lesions)} lesion_ids em comum")

Vazamento treino/val:   0 lesion_ids em comum
Vazamento treino/teste: 0 lesion_ids em comum
Vazamento val/teste:    0 lesion_ids em comum


In [11]:
# Cálculo de pesos de classe para lidar com desbalanceamento
from sklearn.utils.class_weight import compute_class_weight

classes = np.array(sorted(df_train['label'].unique()))
weights = compute_class_weight('balanced', classes=classes, y=df_train['label'])
class_weights = dict(zip(classes.tolist(), weights.tolist()))

print("Class weights:")
inv_label_map = {v: k for k, v in label_map.items()}
for cls, w in class_weights.items():
    print(f"  {inv_label_map[cls]:<8} → {w:.4f}")

Class weights:
  nv       → 0.2133
  mel      → 1.2857
  bkl      → 1.2955
  bcc      → 2.7956
  akiec    → 4.5398
  vasc     → 10.4694
  df       → 11.1522


In [13]:
# Salva os conjuntos de dados e os pesos de classe em arquivos CSV e JSON
df_train.to_csv(PROCESSED_DIR / "train.csv", index=False)
df_val.to_csv(PROCESSED_DIR / "val.csv", index=False)
df_test.to_csv(PROCESSED_DIR / "test.csv", index=False)

with open(PROCESSED_DIR / "class_weights.json", "w") as f:
    json.dump(class_weights, f, indent=2)

print("Arquivos salvos:")
print(f"  train.csv     → {len(df_train)} linhas")
print(f"  val.csv       → {len(df_val)} linhas")
print(f"  test.csv      → {len(df_test)} linhas")
print(f"  class_weights.json")

Arquivos salvos:
  train.csv     → 7182 linhas
  val.csv       → 1306 linhas
  test.csv      → 1527 linhas
  class_weights.json
